In [7]:
"""Definition-based dominating-set solver for large SPECIAL graph families.

No closed-form domination-number formulas are used.

For path, cycle, matching, and edgeless graphs:
    A local greedy algorithm uses N[v] (closed neighborhoods) to select vertices.
For dense special graphs:
    A compact adjacency representation finds vertices or vertex pairs that
    dominate the entire graph without materializing quadratic numbers of edges.

After constructing S, the solver checks a general lower bound, valid for ANY
simple undirected graph: |S| >= ceil(n/(maximum_degree + 1)).
An equal-size dominating set proves its cardinality is exactly gamma(G).

For arbitrary graphs that do not meet the bound, a greedy solution would only
be an upper bound; the program does not incorrectly claim it is minimum.
"""

from collections import deque
from dataclasses import dataclass
from typing import Iterable


@dataclass
class Solution:
    n: int
    graph: str
    size: int
    lower_bound: int
    proven_exact: bool
    first: list[int]
    last: list[int]


# Vertices are represented by 0, 1, ..., n-1 internally.
# (The interface displays 1-based vertex labels.)
def closed_neighborhood(kind: str, n: int, v: int) -> tuple[int, ...]:
    """Return N[v] for the SPARSE graph types only."""
    if kind == "path":
        return tuple(w for w in (v - 1, v, v + 1) if 0 <= w < n)
    if kind == "cycle":
        return ((v - 1) % n, v, (v + 1) % n)
    if kind == "matching":
        partner = v ^ 1
        return (v, partner) if partner < n else (v,)
    if kind == "edgeless":
        return (v,)
    raise ValueError("This graph needs the compact dense-graph solver")


def sparse_greedy(kind: str, n: int, maximum_degree: int) -> Solution:
    """Greedily dominate the graph using the DEFINITION, not a graph formula."""
    dominated = bytearray(n)
    dominated_count = 0
    count = 0
    first = []
    last = deque(maxlen=5)

    for u in range(n):
        if dominated[u]:
            continue

        # Definition: some element of N[u] must be chosen if u is undominated.
        # Choose the candidate covering the most vertices not yet dominated.
        best_vertex = None
        best_gain = -1
        for v in closed_neighborhood(kind, n, u):
            gain = sum(1 for w in closed_neighborhood(kind, n, v)
                       if not dominated[w])
            if gain > best_gain:
                best_gain = gain
                best_vertex = v

        assert best_vertex is not None
        count += 1
        if len(first) < 10:
            first.append(best_vertex + 1)
        last.append(best_vertex + 1)

        for w in closed_neighborhood(kind, n, best_vertex):
            if not dominated[w]:
                dominated[w] = 1
                dominated_count += 1

    assert dominated_count == n, "The selected set does not dominate all vertices."
    lower_bound = (n + maximum_degree) // (maximum_degree + 1)
    return Solution(n, kind, count, lower_bound, count == lower_bound,
                    first, list(last))


def compact_dense_solution(kind: str, n: int, parts: list[int] | None = None) -> Solution:
    """Use compact adjacency rules and verify domination; no edge matrix.

    This is a definition-based certificate search for universal vertices or a
    dominating pair, not a closed-form formula for gamma(G).
    """
    if kind in {"complete", "star", "wheel", "fan"}:
        # These graph definitions give vertex 1 adjacency to every other vertex.
        selected = [1]
        maximum_degree = n - 1
    elif kind in {"bipartite", "multipartite"}:
        assert parts is not None and len(parts) >= 2 and min(parts) >= 1
        # In a complete multipartite graph, vertices in different parts
        # are adjacent; those in the same part are not.
        smallest_size = min(parts)
        maximum_degree = n - smallest_size
        offset = 1
        singleton_vertex = None
        for size in parts:
            if size == 1:
                singleton_vertex = offset
                break
            offset += size

        if singleton_vertex is not None:
            selected = [singleton_vertex]
        else:
            # A vertex in part 1 and one in part 2 dominate every part.
            selected = [1, parts[0] + 1]

        # Verify directly against the complete-multipartite adjacency rule.
        selected_parts = []
        for vertex in selected:
            s = 0
            for part_id, size in enumerate(parts):
                s += size
                if vertex <= s:
                    selected_parts.append(part_id)
                    break
        for part_id in range(len(parts)):
            # Any vertex in a part is dominated by a selected vertex in
            # another part; if the sole selected vertex is its only vertex,
            # membership itself dominates it.
            if any(sp != part_id for sp in selected_parts):
                continue
            if not (parts[part_id] == 1 and part_id in selected_parts):
                raise AssertionError("Candidate is not dominating")
    else:
        raise ValueError("Unsupported graph family")

    lower_bound = (n + maximum_degree) // (maximum_degree + 1)
    return Solution(n, kind, len(selected), lower_bound,
                    len(selected) == lower_bound,
                    selected[:10], selected[-5:])


def solve(kind: str, n: int, parts: list[int] | None = None) -> Solution:
    if not isinstance(n, int) or n < 1:
        raise ValueError("The number of vertices must be a positive integer")
    if kind in {"cycle", "wheel"} and n < (3 if kind == "cycle" else 4):
        raise ValueError("A cycle requires at least 3 vertices, and a wheel at least 4")
    if kind == "fan" and n < 3:
        raise ValueError("A fan requires at least 3 vertices")

    if kind == "path":
        maximum_degree = min(2, n - 1)
    elif kind == "cycle":
        maximum_degree = 2
    elif kind == "matching":
        maximum_degree = 1 if n >= 2 else 0
    elif kind == "edgeless":
        maximum_degree = 0
    else:
        return compact_dense_solution(kind, n, parts)

    return sparse_greedy(kind, n, maximum_degree)


def domination_number():
    families = [
        ("path", "Path graph"),
        ("cycle", "Cycle graph"),
        ("complete", "Complete graph"),
        ("star", "Star graph"),
        ("wheel", "Wheel graph"),
        ("fan", "Fan graph"),
        ("edgeless", "Edgeless graph"),
        ("matching", "Disjoint-edge matching (one isolate if odd)"),
        ("bipartite", "Complete bipartite graph"),
        ("multipartite", "Complete multipartite graph"),
    ]

    print("\nDOMINATION SOLVER — USING THE DEFINITION, NOT FORMULAS\n")
    for i, (_, title) in enumerate(families, 1):
        print(f"{i:2}. {title}")

    try:
        choice = int(input("\nChoose graph type (1-10): ").strip())
        if not 1 <= choice <= len(families):
            raise ValueError("Choose a number from 1 to 10")
        kind = families[choice - 1][0]

        parts = None
        if kind == "bipartite":
            a = int(input("Size of part A: ").strip())
            b = int(input("Size of part B: ").strip())
            if min(a, b) < 1:
                raise ValueError("Both parts must have positive size")
            parts = [a, b]
            n = a + b
        elif kind == "multipartite":
            parts = [int(x) for x in input(
                "Enter part sizes separated by spaces, e.g. 3 4 5: ").split()]
            if len(parts) < 2 or min(parts) < 1:
                raise ValueError("Give at least two positive part sizes")
            n = sum(parts)
        else:
            n = int(input("Enter the number of vertices: ").strip())

        result = solve(kind, n, parts)
    except (ValueError, OverflowError) as exc:
        print("Input error:", exc)
        return

    print("\nRESULTS")
    print("Graph:", families[choice - 1][1])
    print(f"Vertices: {result.n:,}")
    print(f"Size of selected dominating set: {result.size:,}")
    print(f"General lower bound ceil(n / (Delta + 1)): {result.lower_bound:,}")
    if result.proven_exact:
        print(f"EXACT domination number gamma(G): {result.size:,}")
        print("Why exact? A dominating set meets a mathematical lower bound.")
    else:
        print("NOT PROVEN MINIMUM. This is an upper bound, not necessarily gamma(G).")
        print("Use an exact solver (e.g., MILP) or a stronger proof.")
    print("First selected vertices:", result.first)
    print("Last selected vertices:", result.last)

In [9]:
domination_number()


DOMINATION SOLVER — USING THE DEFINITION, NOT FORMULAS

 1. Path graph
 2. Cycle graph
 3. Complete graph
 4. Star graph
 5. Wheel graph
 6. Fan graph
 7. Edgeless graph
 8. Disjoint-edge matching (one isolate if odd)
 9. Complete bipartite graph
10. Complete multipartite graph



Choose graph type (1-10):  8
Enter the number of vertices:  10



RESULTS
Graph: Disjoint-edge matching (one isolate if odd)
Vertices: 10
Size of selected dominating set: 5
General lower bound ceil(n / (Delta + 1)): 5
EXACT domination number gamma(G): 5
Why exact? A dominating set meets a mathematical lower bound.
First selected vertices: [1, 3, 5, 7, 9]
Last selected vertices: [1, 3, 5, 7, 9]
